# 🚀 SageMaker Training Launcher
Run this notebook from **Amazon SageMaker Studio** or a **SageMaker Notebook Instance** (NOT Colab).

**Cost estimate:** ~$2–3 per training run on ml.g4dn.xlarge (~2-3 hours)

### Before running:
1. Confirm you have `HUGGINGFACE_TOKEN` in AWS Secrets Manager (see Cell 1)
2. Confirm your IAM role has SageMaker + S3 + SecretsManager permissions
3. Confirm `ml.g4dn.xlarge` quota is approved (Service Quotas)


In [ ]:
# ── CELL 1: Setup — run this once to store your HF token securely ──
import boto3, json

# Store your HuggingFace token in AWS Secrets Manager (one-time setup)
# This keeps your token secure and out of code.

HF_TOKEN = "hf_YOUR_TOKEN_HERE"   # ← paste your token here, then delete after running

client = boto3.client("secretsmanager", region_name="us-east-1")
try:
    client.create_secret(
        Name="detectai/hf_token",
        SecretString=json.dumps({"HF_TOKEN": HF_TOKEN}),
    )
    print("✅ Secret created: detectai/hf_token")
except client.exceptions.ResourceExistsException:
    client.update_secret(
        SecretId="detectai/hf_token",
        SecretString=json.dumps({"HF_TOKEN": HF_TOKEN}),
    )
    print("✅ Secret updated: detectai/hf_token")

# Clear from memory
HF_TOKEN = ""
print("Token stored securely. Delete it from the cell above now.")


In [ ]:
# ── CELL 2: Install SageMaker SDK + fetch token ────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sagemaker>=2.200.0", "boto3"], check=True)

import boto3, json, sagemaker
from sagemaker import get_execution_role

# Fetch HF token from Secrets Manager
sm_client = boto3.client("secretsmanager", region_name="us-east-1")
secret     = json.loads(sm_client.get_secret_value(SecretId="detectai/hf_token")["SecretString"])
HF_TOKEN   = secret["HF_TOKEN"]
print(f"✅ HF token loaded ({len(HF_TOKEN)} chars)")

# SageMaker session
sess        = sagemaker.Session()
ROLE        = get_execution_role()
BUCKET      = sess.default_bucket()
REGION      = sess.boto_region_name
print(f"Role:   {ROLE.split('/')[-1]}")
print(f"Bucket: {BUCKET}")
print(f"Region: {REGION}")


In [ ]:
# ── CELL 3: Upload training scripts to S3 ──────────────────────
import os, boto3

s3     = boto3.client("s3", region_name=REGION)
PREFIX = "detectai/image-detector/scripts"
FILES  = ["train.py", "requirements.txt"]

for fname in FILES:
    if os.path.exists(fname):
        s3.upload_file(fname, BUCKET, f"{PREFIX}/{fname}")
        print(f"✅ Uploaded {fname} → s3://{BUCKET}/{PREFIX}/{fname}")
    else:
        print(f"⚠️  {fname} not found — make sure it's in the same directory as this notebook")


In [ ]:
# ── CELL 4: Launch SageMaker Training Job ──────────────────────
from sagemaker.pytorch import PyTorch
from datetime import datetime

JOB_NAME = f"detectai-image-{datetime.now().strftime('%Y%m%d-%H%M')}"

estimator = PyTorch(
    entry_point       = "train.py",
    source_dir        = ".",          # uploads train.py + requirements.txt
    role              = ROLE,
    instance_type     = "ml.g4dn.xlarge",   # T4 GPU, ~$0.75/hr
    instance_count    = 1,
    framework_version = "2.2",
    py_version        = "py310",
    hyperparameters   = {
        "epochs":     6,
        "batch_size": 32,
        "lr":         2e-5,
        "hf_token":   HF_TOKEN,
        "hf_repo":    "saghi776/aiscern-image-detector",
    },
    output_path       = f"s3://{BUCKET}/detectai/image-detector/output",
    base_job_name     = "detectai-image",
    keep_alive_period_in_seconds = 300,  # warm pool — saves startup time on reruns
)

print(f"Launching job: {JOB_NAME}")
print(f"Instance:      ml.g4dn.xlarge (~$0.75/hr)")
print(f"Estimated cost: $2-3 total")
print(f"Logs: CloudWatch → /aws/sagemaker/TrainingJobs/{JOB_NAME}\n")

estimator.fit(wait=False, job_name=JOB_NAME)
print(f"✅ Job submitted: {JOB_NAME}")
print(f"Track at: https://console.aws.amazon.com/sagemaker/home#/jobs/{JOB_NAME}")


In [ ]:
# ── CELL 5: Monitor training (run after Cell 4) ────────────────
import boto3, time

sm = boto3.client("sagemaker", region_name=REGION)

# Get the most recent job if JOB_NAME not defined
try:
    job_name = JOB_NAME
except NameError:
    jobs     = sm.list_training_jobs(SortBy="CreationTime", SortOrder="Descending", MaxResults=1)
    job_name = jobs["TrainingJobSummaries"][0]["TrainingJobName"]

print(f"Monitoring: {job_name}")
print("Press Ctrl+C to stop monitoring (job continues running)\n")

while True:
    resp   = sm.describe_training_job(TrainingJobName=job_name)
    status = resp["TrainingJobStatus"]
    
    metrics = {m["MetricName"]: m["Value"] for m in resp.get("FinalMetricDataList", [])}
    elapsed = ""
    if "TrainingStartTime" in resp and "TrainingEndTime" in resp:
        secs    = (resp["TrainingEndTime"] - resp["TrainingStartTime"]).seconds
        elapsed = f" | {secs//60}m {secs%60}s"
    
    cost_est = ""
    if "TrainingStartTime" in resp:
        from datetime import datetime, timezone
        run_hrs  = (datetime.now(timezone.utc) - resp["TrainingStartTime"]).seconds / 3600
        cost_est = f" | ~${run_hrs * 0.75:.2f} so far"
    
    print(f"Status: {status}{elapsed}{cost_est} | Metrics: {metrics}", end="\r")
    
    if status in ("Completed", "Failed", "Stopped"):
        print(f"\n\n{'✅' if status=='Completed' else '❌'} Job {status}")
        if status == "Completed":
            print(f"Model saved to: {resp['ModelArtifacts']['S3ModelArtifacts']}")
            print(f"Check HuggingFace: https://huggingface.co/saghi776/aiscern-image-detector")
        break
    time.sleep(30)
